# 1D Langevin Gaussian

Chapter 18: Langevin Dynamics and Sampling · *Mathematics for AI and Machine Learning* by Fuheng Wu

Convergence of Langevin dynamics to N(0,1) with live histogram.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-18/langevin-1d/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

torch.manual_seed(42)

# The demo's defaults: target N(0, 1) (score -x), step eta, steps, start x0 = 5
eta, steps, x0 = 0.01, 8000, 5.0

# ── Unadjusted Langevin: x <- x + eta * score(x) + sqrt(2 eta) * z ──────────
noise = torch.randn(steps, dtype=torch.float64)
x = x0
trace = torch.empty(steps, dtype=torch.float64)
for k in range(steps):
    x = x + eta * (-x) + math.sqrt(2 * eta) * noise[k].item()
    trace[k] = x

# The mean decays like x0 (1 - eta)^k, so it takes ~ 1/eta steps to forget the start
for k in (0, 100, 300, 1000):
    print(f"step {k:4d}: x = {trace[k].item():+.3f}   expected mean x0 (1 - eta)^k = {x0 * (1 - eta) ** k:+.3f}")

kept = trace[2000:]                                    # discard burn-in
print(f"\nafter burn-in: mean {kept.mean().item():+.3f} (0), variance {kept.var().item():.3f} (target 1)")

# ── Stationary variance of the discretized chain: 2 eta / (1 - (1 - eta)^2) ─
print("exact stationary variance of this chain =", 2 * eta / (1 - (1 - eta) ** 2), "(slightly above 1: step-size bias)")

# Many chains at once show the same thing without a long run
chains = torch.full((20000,), x0, dtype=torch.float64)
for _ in range(1500):
    chains = chains - eta * chains + math.sqrt(2 * eta) * torch.randn_like(chains)
print(f"20000 parallel chains after 1500 steps: mean {chains.mean().item():+.3f}, variance {chains.var().item():.3f}")

step    0: x = +4.992   expected mean x0 (1 - eta)^k = +5.000
step  100: x = +2.537   expected mean x0 (1 - eta)^k = +1.830
step  300: x = +0.595   expected mean x0 (1 - eta)^k = +0.245
step 1000: x = +0.268   expected mean x0 (1 - eta)^k = +0.000

after burn-in: mean -0.019 (0), variance 0.951 (target 1)
exact stationary variance of this chain = 1.0050251256281393 (slightly above 1: step-size bias)


20000 parallel chains after 1500 steps: mean -0.002, variance 1.004
